# QAA Multi-Asset Portfolio Construction

This notebook refactors the original course assignment into a cleaner research workflow.

## Goals

- load and clean the original multi-asset dataset,
- engineer the reduced investment universe,
- study risk-return trade-offs,
- build several portfolio-allocation candidates,
- compare them through historical and simulated metrics.

> Place the original Excel files (`Database_gg3.xlsx` and `Database_gg.xlsx`) inside `data/raw/` before running the notebook end to end.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing 'src/' and 'notebooks/'.")

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import numpy as np
import pandas as pd
from tabulate import tabulate

from qaa.config import PERIODS_PER_YEAR
from qaa.data import load_assignment_data, build_research_universe, compute_log_returns, proxy_equilibrium_weights
from qaa.metrics import (
    annualized_return,
    annualized_volatility,
    annualized_downside_volatility,
    historic_var,
    parametric_var,
    historic_es,
    parametric_es,
    sharpe_ratio,
    sortino_ratio,
    omega_ratio,
    calmar_ratio,
    max_drawdown,
)
from qaa.optimization import (
    portfolio_series,
    long_only_bounds,
    base_constraints,
    group_lower_bound,
    group_upper_bound,
    relative_group_lower_bound,
    solve_min_vol,
    solve_min_downside_vol,
    solve_sortino,
    implied_equilibrium_returns,
    black_litterman_posterior_mean,
    simulate_gbm_paths,
    simulation_log_returns,
)
from qaa.plotting import plot_correlation_heatmap, plot_risk_return_scatter, plot_weights

pd.options.display.float_format = "{:,.4f}".format


## 1. Load the original data

The original script read from hard-coded local Windows paths.  
Here the data-loading logic is centralized in `src/qaa/data.py`.


In [ ]:
data_raw = load_assignment_data()
data_raw.head()

## 2. Baseline asset-universe inspection

In [ ]:
returns_raw = compute_log_returns(data_raw)
rf_annual = returns_raw["US Money Market"].mean() * PERIODS_PER_YEAR
sigma_raw_annual = returns_raw.cov() * PERIODS_PER_YEAR
corr_raw = returns_raw.corr()

asset_vol = returns_raw.std() * np.sqrt(PERIODS_PER_YEAR)
asset_down_vol = annualized_downside_volatility(returns_raw, PERIODS_PER_YEAR)
asset_hist_es = historic_es(returns_raw, PERIODS_PER_YEAR, 0.05)
asset_mu = returns_raw.mean() * PERIODS_PER_YEAR

plot_correlation_heatmap(corr_raw, "Baseline correlation heatmap")
plot_risk_return_scatter(asset_vol, asset_mu, "Risk-return trade-off (baseline universe)", "Volatility", "Annualized return")
plot_risk_return_scatter(asset_down_vol, asset_mu, "Downside-risk trade-off (baseline universe)", "Downside volatility", "Annualized return")
plot_risk_return_scatter(-asset_hist_es, asset_mu, "Expected-shortfall trade-off (baseline universe)", "Historical ES (absolute)", "Annualized return")


## 3. Engineer the reduced research universe

This replicates the original step where the universe is compressed into a cleaner set of investable sleeves:
- blended EM bond,
- blended EU IG bond,
- blended opportunities bucket.


In [ ]:
data = build_research_universe(data_raw)
returns = compute_log_returns(data)

sigma_annual = returns.cov() * PERIODS_PER_YEAR
corr = returns.corr()
mu_annual = returns.mean() * PERIODS_PER_YEAR
down_vol = annualized_downside_volatility(returns, PERIODS_PER_YEAR)
hist_es = historic_es(returns, PERIODS_PER_YEAR, 0.05)
vol_annual = annualized_volatility(returns, PERIODS_PER_YEAR)

plot_correlation_heatmap(corr, "Correlation heatmap (engineered universe)")
plot_risk_return_scatter(vol_annual, mu_annual, "Risk-return trade-off (engineered universe)", "Volatility", "Annualized return")
plot_risk_return_scatter(down_vol, mu_annual, "Downside-risk trade-off (engineered universe)", "Downside volatility", "Annualized return")
plot_risk_return_scatter(-hist_es, mu_annual, "Expected-shortfall trade-off (engineered universe)", "Historical ES (absolute)", "Annualized return")


## 4. Proxy equilibrium returns

In [ ]:
proxy_w = proxy_equilibrium_weights(data)
pi = implied_equilibrium_returns(
    sigma_annual=sigma_annual,
    equilibrium_weights=proxy_w,
    risk_free_rate=rf_annual,
    risk_aversion=4.5,
)

plot_weights(proxy_w, "Proxy equilibrium weights")
plot_risk_return_scatter(vol_annual, pi, "Risk-return trade-off using proxy equilibrium returns", "Volatility", "Implied equilibrium return")
pi.sort_values(ascending=False).to_frame("pi")


## 5. Portfolio constraints

The constrained variants in the original assignment were trying to encode:
- minimum overall European exposure,
- bounded total bond exposure,
- minimum high-yield share inside the bond bucket,
- cap on money-market exposure,
- minimum overall equity exposure,
- minimum EU-equity share inside the equity bucket.


In [ ]:
flag_eu = pd.Series(0, index=returns.columns)
flag_eu[["EU Money Mkt", "EU Bond", "EU Equity", "EU IG Bond"]] = 1

flag_bond = pd.Series(0, index=returns.columns)
flag_bond[["EU Bond", "Global Bond", "EM Bond", "Global Corp. Bond High Yield", "EU IG Bond"]] = 1

flag_hy = pd.Series(0, index=returns.columns)
flag_hy[["Global Corp. Bond High Yield"]] = 1

flag_mm = pd.Series(0, index=returns.columns)
flag_mm[["EU Money Mkt", "US Money Market"]] = 1

flag_equity = pd.Series(0, index=returns.columns)
flag_equity[["EU Equity", "North America Equity", "Pacific Equity", "EM Equity"]] = 1

flag_eu_equity = pd.Series(0, index=returns.columns)
flag_eu_equity[["EU Equity"]] = 1

max_weights_constrained = [0.10, 0.25, 0.30, 0.20, 0.25, 0.25, 0.40, 0.30, 0.30, 0.08, 0.15, 0.35]
bounds_plain = long_only_bounds(returns.columns)
bounds_constrained = long_only_bounds(returns.columns, max_weights=max_weights_constrained)

target_return = 0.10

constraints_plain = base_constraints(returns, target_return=target_return, periods_per_year=PERIODS_PER_YEAR)

constraints_constrained = base_constraints(returns, target_return=target_return, periods_per_year=PERIODS_PER_YEAR) + [
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_eu, 0.35)},
    {"type": "ineq", "fun": group_upper_bound, "args": (flag_bond, 0.35)},
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_bond, 0.25)},
    {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_hy, flag_bond, 0.35)},
    {"type": "ineq", "fun": group_upper_bound, "args": (flag_mm, 0.12)},
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_equity, 0.40)},
    {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_eu_equity, flag_equity, 0.30)},
]


## 6. Solve the optimization problems

In [ ]:
res_markowitz = solve_min_vol(returns, bounds_plain, constraints_plain, PERIODS_PER_YEAR)
res_constrained = solve_min_vol(returns, bounds_constrained, constraints_constrained, PERIODS_PER_YEAR)
res_downside = solve_min_downside_vol(returns, bounds_constrained, constraints_constrained, PERIODS_PER_YEAR)

q = pd.Series([0.03, 0.04, 0.02, 0.30, 0.15], index=range(5))
P = pd.DataFrame(
    [
        [0, 0, 0, 0, 0, -1,  1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0,  0,  1,-1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0,  0, -1, 0, 1, 0, 0, 0],
        [0, 0, 0, 0,-1,  0,  0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 1, -1,  0, 0, 0, 0, 0, 0],
    ],
    columns=returns.columns,
)
confidences = pd.Series([0.50, 0.50, 0.25, 0.25, 0.50], index=P.index)

bl_mu = black_litterman_posterior_mean(
    sigma_annual=sigma_annual,
    pi=pi,
    P=P,
    q=q,
    confidences=confidences,
    tau=1/15,
)

# Use historical covariance but BL posterior mean as target-return input proxy.
returns_bl = returns.copy()
returns_bl.loc[:, :] = returns_bl.values - returns_bl.mean().values + (bl_mu.values / PERIODS_PER_YEAR)

constraints_bl_plain = base_constraints(returns_bl, target_return=target_return, periods_per_year=PERIODS_PER_YEAR)
constraints_bl_constrained = base_constraints(returns_bl, target_return=target_return, periods_per_year=PERIODS_PER_YEAR) + [
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_eu, 0.35)},
    {"type": "ineq", "fun": group_upper_bound, "args": (flag_bond, 0.35)},
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_bond, 0.25)},
    {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_hy, flag_bond, 0.35)},
    {"type": "ineq", "fun": group_upper_bound, "args": (flag_mm, 0.12)},
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_equity, 0.40)},
    {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_eu_equity, flag_equity, 0.30)},
]

res_bl = solve_min_vol(returns_bl, bounds_plain, constraints_bl_plain, PERIODS_PER_YEAR)
res_bl_constrained = solve_min_vol(returns_bl, bounds_constrained, constraints_bl_constrained, PERIODS_PER_YEAR)

constraints_sortino_plain = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
constraints_sortino_constrained = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}] + [
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_eu, 0.35)},
    {"type": "ineq", "fun": group_upper_bound, "args": (flag_bond, 0.35)},
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_bond, 0.25)},
    {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_hy, flag_bond, 0.35)},
    {"type": "ineq", "fun": group_upper_bound, "args": (flag_mm, 0.12)},
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_equity, 0.40)},
    {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_eu_equity, flag_equity, 0.30)},
]

res_sortino = solve_sortino(returns, bounds_plain, constraints_sortino_plain, rf_annual, PERIODS_PER_YEAR)
res_sortino_constrained = solve_sortino(returns, bounds_constrained, constraints_sortino_constrained, rf_annual, PERIODS_PER_YEAR)


In [ ]:
weights = pd.DataFrame(
    {
        "Markowitz": res_markowitz.x,
        "Constrained": res_constrained.x,
        "Downside Constrained": res_downside.x,
        "Black-Litterman": res_bl.x,
        "BL Constrained": res_bl_constrained.x,
        "Sortino": res_sortino.x,
        "Sortino Constrained": res_sortino_constrained.x,
    },
    index=returns.columns,
)

weights.style.format("{:.2%}")


## 7. Inspect portfolio compositions

In [ ]:
for col in weights.columns:
    plot_weights(weights[col], col)

## 8. Historical portfolio return series

In [ ]:
portfolio_returns = pd.DataFrame(
    {name: portfolio_series(returns, weights[name]) for name in weights.columns}
)
portfolio_returns.head()


In [ ]:
ax = (1 + portfolio_returns).cumprod().plot(figsize=(12, 6), grid=True, title="Historical equity curves")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative wealth")


## 9. Forward scenario simulation

In [ ]:
seed = 2800
n_years = 22
n_sims = 1000

simulated_log_returns = {}
for name in portfolio_returns.columns:
    mu_p = portfolio_returns[name].mean() * PERIODS_PER_YEAR
    sigma_p = portfolio_returns[name].std() * np.sqrt(PERIODS_PER_YEAR)
    x0 = float((weights[name] * data.dropna().iloc[-1]).sum())
    paths = simulate_gbm_paths(
        x0=x0,
        mu=mu_p,
        sigma=sigma_p,
        n_years=n_years,
        periods_per_year=PERIODS_PER_YEAR,
        n_sims=n_sims,
        seed=seed,
    )
    simulated_log_returns[name] = simulation_log_returns(paths).median(axis=1)

sim_median = pd.DataFrame(simulated_log_returns)
ax = (1 + sim_median).cumprod().plot(figsize=(12, 6), grid=True, title="Median simulated equity curves")
ax.set_xlabel("Step")
ax.set_ylabel("Cumulative wealth")


## 10. Risk metrics

In [ ]:
risk_metrics = pd.DataFrame(
    {
        "Downside Volatility": annualized_downside_volatility(portfolio_returns, PERIODS_PER_YEAR),
        "Historical VaR (5%)": historic_var(portfolio_returns, PERIODS_PER_YEAR, 0.05),
        "Historical ES (5%)": historic_es(portfolio_returns, PERIODS_PER_YEAR, 0.05),
        "Parametric VaR (5%)": parametric_var(portfolio_returns, PERIODS_PER_YEAR, 0.05),
        "Parametric ES (5%)": parametric_es(portfolio_returns, PERIODS_PER_YEAR, 0.05),
        "Max Drawdown": max_drawdown(portfolio_returns, 3000),
    }
).T

risk_metrics


## 11. Risk-adjusted performance metrics

In [ ]:
risk_adjusted = pd.DataFrame(
    {
        "Sharpe": sharpe_ratio(portfolio_returns, rf_annual, PERIODS_PER_YEAR),
        "Sortino": sortino_ratio(portfolio_returns, rf_annual, PERIODS_PER_YEAR),
        "Omega (10%)": omega_ratio(portfolio_returns, 0.10, PERIODS_PER_YEAR),
        "Calmar": calmar_ratio(portfolio_returns, 252, PERIODS_PER_YEAR),
    }
).T

risk_adjusted


## 12. Final remarks

This notebook keeps the original project idea, but in a format that is much closer to a proper GitHub research repo:

- reusable code in `src/qaa`,
- reproducible relative paths,
- explicit notebook narrative,
- cleaner optimization blocks,
- clearer separation between historical analysis and forward simulation.

The most natural next enhancement would be a **rolling out-of-sample backtest** rather than relying only on full-sample optimization.
